In [7]:
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# =====================================================================
# 1) EDIT: each rater's name and their CSV file(s).
# =====================================================================
RATER_FILES = {
     "Expert1": [
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/Expert/Mark/DUI_MJ - Classifcations.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/Expert/Mark/Hate Crimes_MJ - Classifcations.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/Expert/Mark/Classify_Element_Stolen_Property - Classifcations.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/Expert/Mark/Stalking_Classify_Element - Classifcations.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/Expert/Mark/Classify_Element_Statutory Rape - Classifcations.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/Expert/Mark/Classify_Element_Trespassing - Classifcations.csv',
    ],
    "Expert2": [
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/Expert/Paul/Classify_Element_DUI_2nd - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/Expert/Paul/Hate Crimes_PT - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/Expert/Paul/Received Stolen property_PT - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/Expert/Paul/Stalking_Classify_Element - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/Expert/Paul/Classify_Element_Statutory Rape - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/Expert/Paul/Trespassing_PT - Classifcations.csv',
    ],
    "Gemini 3.1": [
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/LLMs/Google/DUI=gemini-3.1-pro-preview_DUI.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/LLMs/Google/Hate Crimes=gemini-3.1-pro-preview_Hate Crimes.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/LLMs /Google/Received Stolen Property=gemini-3.1-pro-preview_Received Stolen Property.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/LLMs/Google/Stalking=gemini-3.1-pro-preview_Stalking.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/LLMs/Google/Statutory Rape=gemini-3.1-pro-preview_Statutory Rape.csv',
       '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/LLMs/Google/Trespassing=gemini-3.1-pro-preview_Trespassing.csv',
    ],
    "GPT 5.6": [
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/LLMs/OpenAI/DUI=gpt-5.6-sol_DUI.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/LLMs/OpenAI/Hate Crimes=gpt-5.6-sol_Hate Crimes.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/LLMs /OpenAI/Received Stolen Property=gpt-5.6-sol_Received Stolen Property.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/LLMs/OpenAI/Stalking=gpt-5.6-sol_Stalking.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/LLMs/OpenAI/Statutory Rape=gpt-5.6-sol_Statutory Rape.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/LLMs/OpenAI/Trespassing=gpt-5.6-sol_Trespassing.csv',
    ],

    "Lay1": [
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/RAs/Kyle/Classify_Element_DUI_Kyle - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/RAs/Kyle/hate_crimes_kyle - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/RAs/Kyle/Copy of Classify_Element_Recieved_stolen_Kyle - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/RAs/Kyle/Stalking_Kyle - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/RAs/Kyle/Copy of Classify_Element - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/RAs/Kyle/Classify_Element_Trespassing - Classifcations.csv',
    ],
     "Lay2": [
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/DUI/RAs/Sophia/Classify_Element_DUI_Sophia - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Hate Crimes/RAs/Sophia/hate_crimes_Sophia - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Receiving Stolen Propery/RAs/Sophia/Classify_Element_Recieved_stolen_Sophia - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Stalking/RAs/Sophia/Stalking_Sophia_new.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Statutory Rape/RAs/Sophia/Classify_Stautory_Rape_Sophia - Classifcations.csv',
        '/Users/chris/Desktop/IRR CSVs (all 3 agents)/Trespassing/RAs/Sophia/Classify_Element_Trespassing_Sophia - Classifcations.csv',
    ],
}

# =====================================================================
# 2) EDIT: which pairs to compare. use rater's names from above
# =====================================================================
PAIR_LIST = [
    ("Expert1", "Expert2"),
    ("Lay1", "Lay2"),
    ("Expert1", "Gemini 3.1"),
     ("Expert1", "GPT 5.6"),
     ("Expert2", "Gemini 3.1"),
     ("Expert2", "GPT 5.6"),
     ("Lay1", "Gemini 3.1"),
     ("Lay1", "GPT 5.6"),
     ("Lay2", "Gemini 3.1"),
     ("Lay2", "GPT 5.6"),
     ("GPT 5.6", "Gemini 3.1"),
]
#    ...or uncomment the next line to compare EVERY possible pair:
# PAIR_LIST = list(itertools.combinations(RATER_FILES.keys(), 2))

# =====================================================================
# 3) Scale, header variants (no need to edit unless names change)
# =====================================================================
ORDER = [
    "Strict Liability",
    "Negligent",
    "Reckless",
    "Knowing",
    "Purposive",
]
rank_map = {label: i + 1 for i, label in enumerate(ORDER)}

ELEMENT_NAME_OPTIONS = {
    "Action":       ["Action", "Conduct"],
    "Result":       ["Result", "Results"],
    "Circumstance": ["Circumstance", "Circumstances", "Attendant Circumstance"],
}
COLUMN_NUMBERS = [1, 2, 3]


def resolve_cols(df, name_options, numbers=COLUMN_NUMBERS):
    """Return the actual column names present in df for this element."""
    for name in name_options:
        candidate = [f"{name} #{n}" for n in numbers]
        if all(c in df.columns for c in candidate):
            return candidate
    raise KeyError(
        f"None of the naming variants {name_options} found as complete "
        f"'#1/#2/#3' column sets. Available columns: {list(df.columns)}"
    )


def irr_for_min_rank_no_id(r1, r2, name_options, prefix, rank_map, verbose=True):
    r1 = r1.copy()
    r2 = r2.copy()

    cols_r1 = resolve_cols(r1, name_options)
    cols_r2 = resolve_cols(r2, name_options)
    if verbose:
        print(f"({prefix}): using columns {cols_r1} (rater 1) / {cols_r2} (rater 2)")

    def clean_to_na(s):
        s = s.astype("string").str.strip()
        s = s.replace("", pd.NA)
        s = s.replace("No Mental States", pd.NA)
        return s

    rank_cols_r1, rank_cols_r2 = [], []
    for c1, c2 in zip(cols_r1, cols_r2):
        r1[c1] = clean_to_na(r1[c1])
        r2[c2] = clean_to_na(r2[c2])
        rc1, rc2 = f"{prefix}_{c1}_rank", f"{prefix}_{c2}_rank"
        r1[rc1] = r1[c1].map(rank_map)
        r2[rc2] = r2[c2].map(rank_map)
        rank_cols_r1.append(rc1)
        rank_cols_r2.append(rc2)

    r1[f"{prefix}_min_rank"] = r1[rank_cols_r1].min(axis=1, skipna=True)
    r2[f"{prefix}_min_rank"] = r2[rank_cols_r2].min(axis=1, skipna=True)

    merged = pd.DataFrame({
        f"{prefix}_min_rank_r1": r1[f"{prefix}_min_rank"].reset_index(drop=True),
        f"{prefix}_min_rank_r2": r2[f"{prefix}_min_rank"].reset_index(drop=True),
    })

    # (A) NA vs non-NA agreement
    is_na1 = merged[f"{prefix}_min_rank_r1"].isna()
    is_na2 = merged[f"{prefix}_min_rank_r2"].isna()
    na_agreement = (is_na1 == is_na2).mean() * 100
    if is_na1.nunique() > 1 and is_na2.nunique() > 1:
        kappa_na = cohen_kappa_score(is_na1, is_na2)
    else:
        kappa_na = float("nan")

    # (B) Ordinal kappa on ranks, excluding NA rows
    merged_ok = merged.dropna().copy()
    pa = (merged_ok[f"{prefix}_min_rank_r1"] == merged_ok[f"{prefix}_min_rank_r2"]).mean() * 100 if len(merged_ok) else float("nan")
    if len(merged_ok) > 0:
        kappa_linear = cohen_kappa_score(
            merged_ok[f"{prefix}_min_rank_r1"], merged_ok[f"{prefix}_min_rank_r2"],
            weights="linear")
        kappa_quadratic = cohen_kappa_score(
            merged_ok[f"{prefix}_min_rank_r1"], merged_ok[f"{prefix}_min_rank_r2"],
            weights="quadratic")
    else:
        kappa_linear = float("nan")
        kappa_quadratic = float("nan")

    if verbose:
        print(f"({prefix}_min): Aligned rows  {len(merged)} total")
        print(f"({prefix}_min): Used for ordinal kappa  {len(merged_ok)} (both non-missing)")
        print(f"({prefix}_min): Percent agreement on ordinal ranks {pa:.1f}%")
        print(f"({prefix}_min): Weighted kappa (linear) {kappa_linear:.3f}")
        print(f"({prefix}_min): Weighted kappa (quadratic)  {kappa_quadratic:.3f}")
        print(f"({prefix}_min): NA agreement {na_agreement:.1f}%")
        print(f"({prefix}_min): Kappa for NA vs non-NA  {kappa_na:.3f}")

    metrics = {
        "n_total": len(merged),
        "n_used": len(merged_ok),
        "percent_agreement": pa,
        "kappa_linear": kappa_linear,
        "kappa_quadratic": kappa_quadratic,
        "na_agreement": na_agreement,
        "kappa_na": kappa_na,
    }
    return merged, merged_ok, metrics


# =====================================================================
# 4) Load each rater once, then loop over all requested pairs.
# =====================================================================
def load_rater(files, name):
    if not files:
        raise ValueError(f"No files listed for rater '{name}' — add paths in RATER_FILES.")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

rater_data = {name: load_rater(files, name) for name, files in RATER_FILES.items()}

rows = []
for name1, name2 in PAIR_LIST:
    d1, d2 = rater_data[name1], rater_data[name2]
    pair_label = f"{name1} - {name2}"
    print("=" * 60)
    print(f"PAIR: {pair_label}")
    if len(d1) != len(d2):
        print(f"  WARNING: row counts differ ({len(d1)} vs {len(d2)}). "
              f"Rows are aligned by order, so results may be invalid — check the files.")
    for element, name_options in ELEMENT_NAME_OPTIONS.items():
        _, _, m = irr_for_min_rank_no_id(
            d1, d2, name_options=name_options, prefix=element,
            rank_map=rank_map, verbose=True,
        )
        rows.append({"pair": pair_label, "element": element, **m})
    print()

summary = pd.DataFrame(rows)
print("=" * 60)
print("SUMMARY TABLE")
print(summary.round(3).to_string(index=False))

# =====================================================================
# 5) Ready-to-paste PAIRS block for the figure script.
# =====================================================================
def fmt(v, pct=False):
    if pd.isna(v):
        return '"no data"'
    return f"{v:.1f}" if pct else f"{v:.3f}"

element_order = list(ELEMENT_NAME_OPTIONS.keys())
print("\nPAIRS = {")
for pair_label in summary["pair"].unique():
    sub = summary[summary["pair"] == pair_label].set_index("element").reindex(element_order)
    kl = ", ".join(fmt(v) for v in sub["kappa_linear"])
    kq = ", ".join(fmt(v) for v in sub["kappa_quadratic"])
    na = ", ".join(fmt(v, pct=True) for v in sub["na_agreement"])
    n  = ", ".join(str(int(v)) for v in sub["n_used"])
    print(f'    "{pair_label}": {{')
    print(f'        "k_linear":     [{kl}],')
    print(f'        "k_quadratic":  [{kq}],')
    print(f'        "na_agreement": [{na}],')
    print(f'        "n":            [{n}],')
    print(f'    }},')
print("}")

PAIR: Expert1 - Expert2
(Action): using columns ['Action #1', 'Action #2', 'Action #3'] (rater 1) / ['Action #1', 'Action #2', 'Action #3'] (rater 2)
(Action_min): Aligned rows  306 total
(Action_min): Used for ordinal kappa  100 (both non-missing)
(Action_min): Percent agreement on ordinal ranks 82.0%
(Action_min): Weighted kappa (linear) 0.563
(Action_min): Weighted kappa (quadratic)  0.400
(Action_min): NA agreement 73.5%
(Action_min): Kappa for NA vs non-NA  0.473
(Result): using columns ['Result #1', 'Result #2', 'Result #3'] (rater 1) / ['Result #1', 'Result #2', 'Result #3'] (rater 2)
(Result_min): Aligned rows  306 total
(Result_min): Used for ordinal kappa  1 (both non-missing)
(Result_min): Percent agreement on ordinal ranks 100.0%
(Result_min): Weighted kappa (linear) nan
(Result_min): Weighted kappa (quadratic)  nan
(Result_min): NA agreement 87.9%
(Result_min): Kappa for NA vs non-NA  0.045
(Circumstance): using columns ['Circumstance #1', 'Circumstance #2', 'Circumstance 

/opt/homebrew/Caskroom/miniconda/base/envs/natlaw/lib/python3.11/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/natlaw/lib/python3.11/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/opt/homebrew/Caskroom/miniconda/base/envs/natlaw/lib/python3.11/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/natlaw/lib/python3.11/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value enco